# Modelos avançados — rodada de melhorias

Segunda rodada dos modelos avançados, **a partir do que a primeira rodada
(`05_modelos_avancados_kaggle.ipynb`) mostrou**:

- A avaliação em 5 partições deu STab (KS 0.535 ± 0.013) e TabKAN
  (0.531 ± 0.018) à frente de XGB/MLP/GB — são os candidatos com mais
  margem para melhorar.
- Em uma única partição o KS tem erro-padrão ≈ 0.02, então qualquer ganho
  aqui só conta se aparecer também na **avaliação robusta** (seção final).
- TabPFN e Mitra só foram bem com contexto **sem duplicatas** + correção de
  prior — as variantes novas partem dessa configuração.

**Auditoria:** nada da rodada anterior é reexecutado ou sobrescrito. Cada
experimento grava um `model_id` novo e descritivo (o nome diz o modelo e o
que mudou); `checar_id_novo` aborta se o ID já existir em `results/` do
repositório, e o push nunca substitui arquivo já versionado. Todas as
melhorias são **opções novas** nos wrappers de `src/churn_telecom/models/`,
com padrões que reproduzem exatamente a rodada anterior.

**No Kaggle, antes de rodar:** *Settings → Internet → On* e
*Accelerator → GPU*; depois **"Save Version" → "Save & Run All (Commit)"**.


In [ ]:
#@title Setup (clona o repo e instala o pacote)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jpbezerra/redes-neurais.git"   # ajuste se necessario
BRANCH   = "feat/churn"   # ajuste aqui se a branch de trabalho mudar
REPO_DIR = "redes-neurais"
SUBPATH  = "projeto/churn-telecom"

def _sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=False)

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    base = Path("/content") if IN_COLAB else Path("/kaggle/temp")  # fora de /kaggle/working: o clone nao entra no Output
    root = base / REPO_DIR
    if root.exists():
        _sh(f"cd {root} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
    else:
        _sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {root}")
    PROJECT_ROOT = root / SUBPATH
    _sh(f"pip install -q -e {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd().parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# No Kaggle, tudo que for gerado vai para /kaggle/working/saida (so isso aparece
# no Output, em vez do repositorio inteiro).
OUT_DIR = Path("/kaggle/working/saida") if IN_KAGGLE else PROJECT_ROOT

DATA_DIR    = PROJECT_ROOT / "data"
RESULTS_DIR = OUT_DIR / "results"
TABLES_DIR  = OUT_DIR / "reports" / "tables"
for d in (DATA_DIR, RESULTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)


## Instalação dos modelos avançados

In [ ]:
!pip install -q tabpfn pykan tabkan
!pip install -q "autogluon.tabular[mitra]"


## Preparação dos dados

Mesma partição da rodada anterior (`SEED = 0`), então as probabilidades
já salvas (`predicoes.npz`) dos modelos antigos se alinham linha a linha com
as dos novos — dá para combiná-los em ensemble sem retreinar nada.


In [ ]:
import torch
import joblib
import numpy as np
import pandas as pd

from churn_telecom.data import load_raw, split_three_stage, assert_sem_vazamento
from churn_telecom.features import Preprocessor
from churn_telecom.metrics import full_report
from churn_telecom.checkpointing import save_run, load_all_metadata
from churn_telecom.ensemble import corrigir_prior, salvar_predicoes, media_probabilidades

SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

df = load_raw(DATA_DIR)
split = split_three_stage(df, seed=SEED)
assert_sem_vazamento(split)
df_train, df_val, df_test = df.iloc[split.train_idx], df.iloc[split.val_idx], df.iloc[split.test_idx]

prep = Preprocessor().fit(df_train)
X_train, cols = prep.transform(df_train)
X_val, _ = prep.transform(df_val)
X_test, _ = prep.transform(df_test)
y_train = prep.transform_target(df_train)
y_val = prep.transform_target(df_val)
y_test = prep.transform_target(df_test)

# Versao SEM linhas duplicadas do treino (mesmas linhas originais, sem oversampling).
# TabPFN/Mitra fazem in-context learning: linhas repetidas viram memorizacao e
# distorcem a calibracao, entao eles tambem sao treinados nesta versao.
df_train_nat = df.iloc[split.train_no_oversample_idx]
X_train_nat, _ = prep.transform(df_train_nat)
y_train_nat = prep.transform_target(df_train_nat)
PRIOR_NAT = float(y_train_nat.mean())

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"X_train_nat (sem duplicatas): {X_train_nat.shape} | prior de churn: {PRIOR_NAT:.3f}")


## Utilitários

Igual à rodada anterior (`rodar` isola falhas por modelo), mais:

- `checar_id_novo`: aborta o experimento se o `model_id` já existir no
  Output **ou** em `results/` do repositório clonado (nunca sobrescreve
  a auditoria);
- `salvar`: grava também KS/AUROC de **validação** no `metadata.json`, para
  que a escolha entre variantes (feita só na validação) fique registrada.


In [ ]:
import traceback

from churn_telecom.metrics import auroc
from churn_telecom.ensemble import media_logits

FALHAS = {}
RESULTS_REPO = PROJECT_ROOT / "results"   # resultados ja versionados (rodadas anteriores)

def checar_id_novo(model_id):
    for base in (RESULTS_DIR, RESULTS_REPO):
        if (base / model_id).exists():
            raise FileExistsError(f"{model_id} ja existe em {base} — use um ID novo (auditoria).")

def metricas_validacao(p_val):
    return {"ks_validacao": full_report(y_val, p_val)["ks"], "auroc_validacao": auroc(y_val, p_val)}

def salvar(model_id, prever, corrigir=None, **kwargs):
    # prever(X) -> probabilidades; corrigir ajusta o prior (TabPFN/Mitra sem duplicatas)
    pv, pt = prever(X_val), prever(X_test)
    if corrigir is not None:
        pv, pt = corrigir(pv), corrigir(pt)
    metricas = full_report(y_test, pt)
    val = metricas_validacao(pv)
    print(model_id, "| val:", {k: round(v, 4) for k, v in val.items()},
          "| teste:", {k: round(metricas[k], 4) for k in ("ks", "auroc", "recall", "precision")})
    save_run(RESULTS_DIR, model_id, metrics=metricas, dataset_info={"metricas_validacao": val}, **kwargs)
    salvar_predicoes(RESULTS_DIR, model_id, pv, pt)
    return val

def rodar(nome, fn):
    try:
        fn()
        print(f"[ok] {nome}")
    except Exception as e:
        FALHAS[nome] = f"{type(e).__name__}: {e}"
        print(f"[FALHOU] {nome}: {type(e).__name__}: {e}")
        traceback.print_exc()

VALIDACAO = {}   # model_id -> metricas de validacao (para escolher variantes sem olhar o teste)


## STab — melhorias de treino e de embedding

Rodada anterior: `stab_churn_dim32_depth2_heads4_oversample` (embedding
compartilhado + posicional, parada pela cross-entropy de validação).

Mudanças testadas (opções novas em `models/stab.py`):

- **embedding por feature** (`W_i·x + b_i`, como no FT-Transformer): cada
  feature ganha sua própria projeção, em vez de uma projeção única
  diferenciada só por um vetor posicional somado;
- **parada e escolha da época pelo AUROC de validação** — o KS só depende
  da ordem dos scores, então o critério de ranking é mais alinhado do que
  a cross-entropy (que também premia calibração);
- **ReduceLROnPlateau** + gradient clipping + `weight_decay` maior.

Duas capacidades: a mesma do original e uma maior. A melhor (pelo AUROC de
**validação**) é depois re-treinada com 5 sementes e as probabilidades são
médias (*bagging de sementes*) — redes pequenas em dados tabulares variam
bastante entre sementes, e a média reduz essa variância.


In [ ]:
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab

MELHORIAS_TREINO = dict(embedding="por_feature", early_stop_metric="val_auroc",
                        scheduler="plateau", max_grad_norm=1.0, patience=20)

CONFIGS_STAB = {
    "stab_churn_dim32_depth2_heads4_embfeature_parada_auroc_plateau":
        STabHyperparams(dim=32, depth=2, heads=4, attn_dropout=0.1, ff_dropout=0.1, lr=1e-3,
                        weight_decay=1e-4, batch_size=64, max_epochs=200, seed=SEED, **MELHORIAS_TREINO),
    "stab_churn_dim64_depth3_heads8_embfeature_parada_auroc_plateau":
        STabHyperparams(dim=64, depth=3, heads=8, attn_dropout=0.15, ff_dropout=0.15, lr=5e-4,
                        weight_decay=1e-4, batch_size=128, max_epochs=200, seed=SEED, **MELHORIAS_TREINO),
}

def experimento_stab(model_id, hp):
    checar_id_novo(model_id)
    modelo, historico = treinar_stab(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    VALIDACAO[model_id] = salvar(
        model_id, lambda X: prever_stab(modelo, X, device=device),
        framework="PyTorch (STab — implementacao propria)", hyperparameters=hp.__dict__,
        architecture={"dim": hp.dim, "depth": hp.depth, "heads": hp.heads, "embedding": hp.embedding},
        history=historico, notes="STab com embedding por feature, parada pelo AUROC de validacao e ReduceLROnPlateau.",
        save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

for mid, hp in CONFIGS_STAB.items():
    rodar(mid, lambda mid=mid, hp=hp: experimento_stab(mid, hp))


In [ ]:
from dataclasses import replace

N_SEMENTES_BAGGING = 5

def melhor_por_validacao(ids):
    candidatos = [i for i in ids if i in VALIDACAO]
    if not candidatos:
        raise RuntimeError(f"nenhuma variante concluiu: {ids}")
    return max(candidatos, key=lambda i: VALIDACAO[i]["auroc_validacao"])

def experimento_stab_bagging():
    base_id = melhor_por_validacao(CONFIGS_STAB)
    model_id = f"{base_id}_bagging{N_SEMENTES_BAGGING}sementes"
    checar_id_novo(model_id)
    print("config escolhida pela validacao:", base_id)
    modelos = []
    for k in range(N_SEMENTES_BAGGING):
        hp = replace(CONFIGS_STAB[base_id], seed=SEED + k)
        m, _ = treinar_stab(X_train, y_train, X_val, y_val, hp, device=device)
        modelos.append(m)
    prever = lambda X: np.mean([prever_stab(m, X, device=device) for m in modelos], axis=0)
    def salvar_pesos(d):
        for k, m in enumerate(modelos):
            torch.save(m.state_dict(), d / f"model_seed{SEED + k}.pt")
    VALIDACAO[model_id] = salvar(
        model_id, prever, framework="PyTorch (STab — implementacao propria)",
        hyperparameters={**CONFIGS_STAB[base_id].__dict__, "seeds": [SEED + k for k in range(N_SEMENTES_BAGGING)]},
        architecture={"config_base": base_id}, save_model_fn=salvar_pesos,
        notes=f"Media das probabilidades de {N_SEMENTES_BAGGING} sementes da config {base_id} (escolhida pelo AUROC de validacao).")

rodar("stab_bagging", experimento_stab_bagging)


## TabKAN — mesmas melhorias de treino, mais largura

Rodada anterior: `tabkan_churn_chebyshev_largura10_grau3_oversample`.
Aqui: parada pelo AUROC de validação + ReduceLROnPlateau + clipping, na
mesma largura e numa camada oculta mais larga com `weight_decay` maior.


In [ ]:
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

MELHORIAS_TABKAN = dict(early_stop_metric="val_auroc", scheduler="plateau", max_grad_norm=1.0, patience=20)

CONFIGS_TABKAN = {
    "tabkan_churn_chebyshev_largura10_grau3_parada_auroc_plateau":
        TabKANHyperparams(hidden_width=10, degree=3, lr=1e-3, weight_decay=1e-5, batch_size=64,
                          max_epochs=300, seed=SEED, **MELHORIAS_TABKAN),
    "tabkan_churn_chebyshev_largura32_grau3_wd1e-4_parada_auroc_plateau":
        TabKANHyperparams(hidden_width=32, degree=3, lr=1e-3, weight_decay=1e-4, batch_size=64,
                          max_epochs=300, seed=SEED, **MELHORIAS_TABKAN),
}

def experimento_tabkan(model_id, hp):
    checar_id_novo(model_id)
    modelo, historico = treinar_tabkan(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    VALIDACAO[model_id] = salvar(
        model_id, lambda X: prever_tabkan(modelo, X, device=device),
        framework="tabkan (ChebyshevKAN)", hyperparameters=hp.__dict__,
        architecture={"hidden_width": hp.hidden_width, "degree": hp.degree}, history=historico,
        notes="TabKAN com parada pelo AUROC de validacao e ReduceLROnPlateau.",
        save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

for mid, hp in CONFIGS_TABKAN.items():
    rodar(mid, lambda mid=mid, hp=hp: experimento_tabkan(mid, hp))


## KAN (pykan) — grade mais fina e regularização

Rodada anterior: `kan_churn_largura8_grid3_k3_oversample` (o mais fraco
dos modelos neurais, KS 0.506). Grade de splines mais fina (`grid=5`),
`weight_decay` e o mesmo critério de parada/scheduler das outras redes.


In [ ]:
from churn_telecom.models.kan import KANHyperparams, treinar_kan, prever_kan

def experimento_kan():
    model_id = "kan_churn_largura8_grid5_k3_wd1e-4_parada_auroc_plateau"
    checar_id_novo(model_id)
    hp = KANHyperparams(hidden_width=8, grid=5, k=3, lr=3e-3, batch_size=128, max_epochs=150, patience=15,
                        weight_decay=1e-4, early_stop_metric="val_auroc", scheduler="plateau", seed=SEED)
    modelo, historico = treinar_kan(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    VALIDACAO[model_id] = salvar(
        model_id, lambda X: prever_kan(modelo, X, device=device),
        framework="pykan (KAN)", hyperparameters=hp.__dict__,
        architecture={"hidden_width": hp.hidden_width, "grid": hp.grid, "k": hp.k},
        history=pd.DataFrame(historico), notes="KAN com grid=5, weight decay e parada pelo AUROC de validacao.",
        save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))

rodar("kan", experimento_kan)


## TabPFN v2 — contexto sem duplicatas, ensemble interno maior

Rodada anterior: melhor configuração foi
`tabpfn_v2_churn_contexto_original_prior_corrigido` (16 estimadores
internos). Aqui: 32 estimadores (mais permutações de features/classes no
ensemble interno do TabPFN — mais estável, sem risco de overfitting porque
não há treino).


In [ ]:
from churn_telecom.models.tabpfn import TabPFNHyperparams, treinar_tabpfn, prever_tabpfn

def experimento_tabpfn():
    model_id = "tabpfn_v2_churn_contexto_original_prior_corrigido_32estimadores"
    checar_id_novo(model_id)
    hp = TabPFNHyperparams(device=device, n_estimators=32, versao="v2", seed=SEED)
    modelo = treinar_tabpfn(X_train_nat, y_train_nat, hp)
    VALIDACAO[model_id] = salvar(
        model_id, lambda X: prever_tabpfn(modelo, X), corrigir=lambda p: corrigir_prior(p, PRIOR_NAT),
        framework="TabPFN v2",
        hyperparameters={**hp.__dict__, "n_linhas_contexto": len(X_train_nat), "prior_corrigido": PRIOR_NAT},
        notes="TabPFN v2, contexto sem duplicatas, 32 estimadores internos.",
        save_model_fn=lambda d: None)

rodar("tabpfn", experimento_tabpfn)


## Mitra — contexto sem duplicatas, ensemble interno e mais fine-tuning

Rodada anterior: `mitra_churn_contexto_original_prior_corrigido` (KS 0.533,
o melhor modelo isolado na partição única), com os padrões do AutoGluon
(`n_estimators=1`, fine-tuning de 50 passos). Aqui: 4 estimadores internos
e 100 passos de fine-tuning.


In [ ]:
from churn_telecom.models.mitra import MitraHyperparams, treinar_mitra, prever_mitra, salvar_mitra

def experimento_mitra():
    model_id = "mitra_churn_contexto_original_prior_corrigido_4estimadores_finetune100"
    checar_id_novo(model_id)
    hp = MitraHyperparams(time_limit_segundos=1200, num_gpus=1 if device == "cuda" else 0, seed=SEED,
                          n_estimators=4, fine_tune_steps=100)
    preditor = treinar_mitra(X_train_nat, y_train_nat, cols, hp, path=f"/kaggle/temp/ag_{model_id}")
    VALIDACAO[model_id] = salvar(
        model_id, lambda X: prever_mitra(preditor, X, cols), corrigir=lambda p: corrigir_prior(p, PRIOR_NAT),
        framework="AutoGluon (extra [mitra])",
        hyperparameters={**hp.__dict__, "n_linhas_contexto": len(X_train_nat), "prior_corrigido": PRIOR_NAT},
        notes="Mitra, contexto sem duplicatas, 4 estimadores internos, 100 passos de fine-tuning.",
        save_model_fn=lambda d: salvar_mitra(preditor, d / "autogluon_predictor"))

rodar("mitra", experimento_mitra)


## XGBoost — mais árvores com early stopping na validação

Os baselines de boosting usam 100 árvores fixas, e a busca Optuna anterior
sobreajustou à validação (20 trials × 5 hiperparâmetros). Aqui não há
busca: uma configuração regularizada padrão de mercado (lr baixo,
subsample/colsample 0.8, `min_child_weight`, L2) e **só o número de
árvores** é escolhido pela validação, via early stopping no AUROC.


In [ ]:
from churn_telecom.models.boosting import XGBoostHyperparams, treinar_xgboost

HP_XGB_MELHORADO = XGBoostHyperparams(learning_rate=0.03, n_estimators=2000, max_depth=4, subsample=0.8,
                                      colsample_bytree=0.8, min_child_weight=5, reg_lambda=5.0,
                                      eval_metric="auc", early_stopping_rounds=100, seed=SEED)

def experimento_xgb():
    model_id = "xgb_churn_lr003_prof4_sub08_col08_earlystop_auc_validacao"
    checar_id_novo(model_id)
    modelo = treinar_xgboost(X_train, y_train, HP_XGB_MELHORADO, X_val=X_val, y_val=y_val)
    print("arvores escolhidas pela validacao:", modelo.best_iteration + 1)
    VALIDACAO[model_id] = salvar(
        model_id, lambda X: modelo.predict_proba(X)[:, 1], framework="XGBoost",
        hyperparameters={**HP_XGB_MELHORADO.__dict__, "n_arvores_escolhidas": int(modelo.best_iteration + 1)},
        notes="XGBoost regularizado; numero de arvores por early stopping no AUROC de validacao.",
        save_model_fn=lambda d: joblib.dump(modelo, d / "model.joblib"))

rodar("xgb", experimento_xgb)


## Ensembles — modelos novos + modelos já salvos da rodada anterior

A partição é a mesma, então as `predicoes.npz` já versionadas da rodada
anterior entram como candidatas sem retreino. Mudanças em relação aos
ensembles anteriores:

- **média no espaço logit** (média geométrica das odds) em vez da média de
  probabilidades — modelos com escalas diferentes (prior corrigido vs.
  treino balanceado) não são achatados;
- seleção dos membros pelo **AUROC de validação** (menos ruidoso que o KS
  de validação), com `k` fixado antes de olhar qualquer resultado.
  O teste não participa da escolha.


In [ ]:
def construir_ensembles():
    fontes = {}
    for base in (RESULTS_REPO, RESULTS_DIR):   # anteriores + desta rodada
        for d in sorted(base.iterdir()):
            if (d / "predicoes.npz").exists() and not d.name.startswith("ensemble_"):
                fontes[d.name] = d / "predicoes.npz"
    P = {i: np.load(f) for i, f in fontes.items()}
    for i in P:
        assert len(P[i]["p_val"]) == len(y_val) and len(P[i]["p_test"]) == len(y_test), f"{i}: particao diferente"
    auc_val = {i: auroc(y_val, P[i]["p_val"]) for i in P}
    ordem = sorted(P, key=lambda i: -auc_val[i])
    print("AUROC de validacao:", {i: round(auc_val[i], 4) for i in ordem})
    novos = [i for i in ordem if (RESULTS_DIR / i).exists()]
    for model_id, membros in [("ensemble_media_logit_top5_validacao_churn", ordem[:5]),
                              ("ensemble_media_logit_modelos_melhorados_churn", novos)]:
        checar_id_novo(model_id)
        if len(membros) < 2:
            print(f"{model_id}: menos de 2 membros, pulando"); continue
        pv = media_logits([P[m]["p_val"] for m in membros])
        pt = media_logits([P[m]["p_test"] for m in membros])
        metricas = full_report(y_test, pt)
        val = metricas_validacao(pv)
        print(model_id, membros, "| val:", {k: round(v, 4) for k, v in val.items()},
              "| teste:", {k: round(metricas[k], 4) for k in ("ks", "auroc", "recall", "precision")})
        save_run(RESULTS_DIR, model_id, framework="media no espaco logit", hyperparameters={"membros": membros},
                 metrics=metricas, dataset_info={"metricas_validacao": val},
                 notes="Ensemble por media dos logits; membros escolhidos pelo AUROC de validacao.",
                 save_model_fn=lambda d: None)
        salvar_predicoes(RESULTS_DIR, model_id, pv, pt)

rodar("ensembles", construir_ensembles)


## Avaliação robusta (5 partições) — antigo vs. melhorado, lado a lado

Mesmas 5 sementes de partição da rodada anterior (0–4). As versões
**originais** de STab/TabKAN/XGB são re-treinadas aqui também, para que a
comparação seja pareada (mesma partição, mesmo hardware) — os números da
rodada anterior continuam intactos em
`reports/tables/avaliacao_robusta_resumo_kaggle.csv`.

A melhoria só deve ser considerada real se `ks_mean` subir com folga em
relação ao desvio (`ks_std`).


In [ ]:
from churn_telecom.robustez import avaliar_multiplas_particoes, resumir
from churn_telecom.ensemble import media_probabilidades
from churn_telecom.models.boosting import GradientBoostingHyperparams, treinar_gradient_boosting

STAB_ESCOLHIDO = melhor_por_validacao(CONFIGS_STAB) if any(i in VALIDACAO for i in CONFIGS_STAB) else None
TABKAN_ESCOLHIDO = melhor_por_validacao(CONFIGS_TABKAN) if any(i in VALIDACAO for i in CONFIGS_TABKAN) else None
print("STab escolhido:", STAB_ESCOLHIDO, "| TabKAN escolhido:", TABKAN_ESCOLHIDO)

def f_stab_original(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_stab(Xtr, ytr, Xva, yva, STabHyperparams(seed=sd), device=device)
    return lambda X: prever_stab(m, X, device=device)

def f_stab_melhorado(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_stab(Xtr, ytr, Xva, yva, replace(CONFIGS_STAB[STAB_ESCOLHIDO], seed=sd), device=device)
    return lambda X: prever_stab(m, X, device=device)

def f_stab_melhorado_bagging(Xtr, ytr, Xva, yva, sd):
    ms = [treinar_stab(Xtr, ytr, Xva, yva, replace(CONFIGS_STAB[STAB_ESCOLHIDO], seed=100 * sd + k),
                       device=device)[0] for k in range(N_SEMENTES_BAGGING)]
    return lambda X: np.mean([prever_stab(m, X, device=device) for m in ms], axis=0)

def f_tabkan_original(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_tabkan(Xtr, ytr, Xva, yva, TabKANHyperparams(seed=sd), device=device, verbose=False)
    return lambda X: prever_tabkan(m, X, device=device)

def f_tabkan_melhorado(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_tabkan(Xtr, ytr, Xva, yva, replace(CONFIGS_TABKAN[TABKAN_ESCOLHIDO], seed=sd),
                          device=device, verbose=False)
    return lambda X: prever_tabkan(m, X, device=device)

def f_xgb_original(Xtr, ytr, Xva, yva, sd):
    m = treinar_xgboost(Xtr, ytr, XGBoostHyperparams(seed=sd))
    return lambda X: m.predict_proba(X)[:, 1]

def f_xgb_melhorado(Xtr, ytr, Xva, yva, sd):
    m = treinar_xgboost(Xtr, ytr, replace(HP_XGB_MELHORADO, seed=sd), X_val=Xva, y_val=yva)
    return lambda X: m.predict_proba(X)[:, 1]

def avaliacao_robusta():
    fabricas = {"stab_original": f_stab_original, "tabkan_original": f_tabkan_original,
                "xgb_original": f_xgb_original, "xgb_melhorado": f_xgb_melhorado}
    if STAB_ESCOLHIDO:
        fabricas.update({"stab_melhorado": f_stab_melhorado, "stab_melhorado_bagging": f_stab_melhorado_bagging})
    if TABKAN_ESCOLHIDO:
        fabricas["tabkan_melhorado"] = f_tabkan_melhorado
    ensembles = {
        "ensemble_media_prob_originais": (["stab_original", "tabkan_original", "xgb_original"], media_probabilidades),
        "ensemble_media_logit_melhorados": (["stab_melhorado_bagging", "tabkan_melhorado", "xgb_melhorado"], media_logits),
    }
    longo = avaliar_multiplas_particoes(df, fabricas, seeds=range(5), incluir_ensemble=False, ensembles=ensembles)
    longo.to_csv(TABLES_DIR / "avaliacao_robusta_modelos_melhorados_por_particao.csv", index=False)
    resumo = resumir(longo)
    resumo.to_csv(TABLES_DIR / "avaliacao_robusta_modelos_melhorados_resumo.csv", index=False)
    print(resumo.round(4).to_string())

rodar("avaliacao_robusta", avaliacao_robusta)


## Comparação — rodada anterior vs. esta rodada (partição única)

Lembre que, numa partição só, diferenças de KS abaixo de ~0.02 estão
dentro do ruído; a tabela da avaliação robusta acima é a que decide.


In [ ]:
pd.DataFrame([{"modelo": k, "erro": v} for k, v in FALHAS.items()],
             columns=["modelo", "erro"]).to_csv(TABLES_DIR / "modelos_melhorados_falhas.csv", index=False)
print("Falhas:", FALHAS or "nenhuma")

antes = load_all_metadata(RESULTS_REPO).assign(rodada="anterior")
agora = load_all_metadata(RESULTS_DIR).assign(rodada="melhorias")
cols = ["rodada", "model_id", "metrics.ks", "metrics.auroc", "metrics.recall", "metrics.precision"]
comparacao = pd.concat([antes, agora], ignore_index=True)
comparacao = comparacao[[c for c in cols if c in comparacao.columns]].sort_values("metrics.ks", ascending=False)
comparacao.to_csv(TABLES_DIR / "modelos_melhorados_comparacao_particao_unica.csv", index=False)
comparacao.reset_index(drop=True)


## Salvando os resultados

Igual à rodada anterior (zip no Output + push opcional com o segredo
`GITHUB_TOKEN`), mas para a branch **`results/churn-modelos-melhorados`** e
com uma trava extra: se algum arquivo já existir no repositório clonado, o
push é abortado em vez de sobrescrever. Localmente:
`git fetch origin results/churn-modelos-melhorados` e
`git checkout origin/results/churn-modelos-melhorados -- projeto/churn-telecom/results projeto/churn-telecom/reports/tables`.


In [ ]:
import shutil

RESULTS_BRANCH = "results/churn-modelos-melhorados"
LIMITE_MB = 20   # GitHub rejeita >100 MB por arquivo (o preditor do Mitra tem ~290 MB)

if IN_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/results_churn_modelos_melhorados", "zip", OUT_DIR)
    print("Zip gerado:", zip_path)

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None

    if not token:
        print("Sem segredo GITHUB_TOKEN: pulando o push (baixe o zip pelo Output).")
    else:
        try:
            def copiar_pequenos(origem, destino, pulados):
                for f in Path(origem).rglob("*"):
                    if not f.is_file():
                        continue
                    if f.stat().st_size > LIMITE_MB * 1024 * 1024:
                        pulados.append(f"{f.relative_to(origem)} ({f.stat().st_size / 1e6:.0f} MB)")
                        continue
                    alvo = Path(destino) / f.relative_to(origem)
                    if alvo.exists():   # auditoria: nunca sobrescreve resultado/tabela ja versionado
                        raise FileExistsError(f"{alvo} ja existe no repositorio — push abortado.")
                    alvo.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, alvo)

            pulados = []
            copiar_pequenos(RESULTS_DIR, PROJECT_ROOT / "results", pulados)
            copiar_pequenos(TABLES_DIR, PROJECT_ROOT / "reports" / "tables", pulados)
            if pulados:
                print(f"Arquivos > {LIMITE_MB} MB NAO enviados ao GitHub (estao no zip):")
                for x in pulados:
                    print("  -", x)

            def git(*args):
                r = subprocess.run(["git", "-C", str(root), "-c", "user.name=kaggle-run",
                                    "-c", "user.email=kaggle-run@users.noreply.github.com", *args],
                                   capture_output=True, text=True)
                saida = (r.stdout + r.stderr).replace(token, "***")
                if r.returncode != 0:
                    raise RuntimeError(f"git {args[0]} falhou: {saida}")
                return saida

            git("checkout", "-B", RESULTS_BRANCH)
            git("add", "-f", f"{SUBPATH}/results", f"{SUBPATH}/reports/tables")
            git("commit", "-m", "results: saidas do notebook de modelos melhorados (Kaggle)")
            url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            git("push", "--force", url, f"HEAD:refs/heads/{RESULTS_BRANCH}")
            print(f"Resultados enviados para a branch {RESULTS_BRANCH}.")
        except Exception as e:
            print("Push falhou (o zip continua valido):", str(e)[:600])
